Capstone-2: Banking Customer Support AI Agent using Multi-Agent Architecture.<br>

Problem scenario:<br>

Modern digital banking platforms handle a high volume of customer service interactions, often through fragmented systems that struggle to personalize responses or provide timely status updates. The need for scalable, intelligent support systems has led to the emergence of AI-driven agents capable of parsing user sentiment, managing service records, and handling real-time queries. <br>
 
This project explores the development of a multi-agent GenAI system tailored for banking customer support workflows. The goal is to reduce manual effort, enhance customer satisfaction, and ensure timely response to support-related feedback and queries.

Project objective:<br>

This project aims to build: 
- Classification of incoming user messages into feedback (positive or 
negative) or queries 
- Personalized responses based on classification and user sentiment 
- Ticket tracking and updates through integration with a support database.


Solution Steps: <br>
1. Used two Langchain Agent
2. First Agent decides if the Fedback in Positive, Negative or a Query to check Ticket Status
3. If the feedback is Positive, appropriate Thank you Response is formatted and send back.
4. If the feedback is Negative, Agent asks if a ticket needs to be created.
5. If a new ticket needs to be created then, it goes ahead and creats a Ticket.
6. Primary key is User Name that is captured at the begining when Custmer Suppoer Agent come online  
7. If the User Input is a Query to List Tickets or Get Status of Ticket, then request is passed to a seconf Agent that make ToolCall to either List Ticket Tool or Get Status Tool.
8. All response of interactions are shown in the Test Case below
9. Models use for Interactions are - "llama3.1:8b-instruct-q4_K_M"

Test Cases: <br>
User Input:  I am happy to get my firs Credit Card - Thank you <br>
Assistant:  Thank you for the PositiveFeedback

User Input:  Can i get all tickets in my name ? <br>
Inside List User Tickets <br>
Assistant:  Number of Tickets for User: Menon - [471718, 299247]

User Input:  What is status of ticket number 299247 ? <br>
Inside Ticket Status Lookup <br>
Assistant:  Status of your ticket #299247 is currently marked as: CLOSED.

User Input:  Wha about 471718 ? <br>
Inside Ticket Status Lookup <br>
Assistant:  Status of your ticket #471718 is currently marked as: OPEN.

User Input:  I was not happy with he Teler Interaction when i visited back last week. I need to open a case. <br>
Assistant:  Sorry to hear the Feedback, Should i create a Support Ticket to resolve your concern ? (Y/N) <br>
Assistant:  We apologize for the inconvenience. A new ticket #128704 has been generated, and our team will follow up shortly. <br>


In [6]:
# Import System Packages
import os
import keyboard
import json
import random
from dotenv import load_dotenv

# Import SQLLite
import sqlite3

# Import Langchain Agents Modules.
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage


# Import Langchain Ollama Chatbot Interfaces
from langchain_ollama import ChatOllama

# Import Sentence Embedding Transformer
import textwrap as tw
from sentence_transformers import SentenceTransformer, CrossEncoder

# Import RAG DB FAISS
import faiss
# from llama_parse import LlamaParse
from llama_cloud import LlamaCloud
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter   # MarkdownNodeParser is efficient if expand[..] is markdown  
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore


# Huggingface Cache Folder
hf_cache_folder = os.path.expanduser("~/.cache/huggingface/hub")

# Connect to SQLLite DB Path
sqllite_db_file = "C:\\RanjithC\\AIProjects\\PromptEngg\\huggingface\\data\\simpli_learn\\custsupport\customer_support.db"

# Load Env Info
load_dotenv()

<>:39: SyntaxWarning: invalid escape sequence '\c'
<>:39: SyntaxWarning: invalid escape sequence '\c'
C:\Users\renme\AppData\Local\Temp\ipykernel_52084\1181723146.py:39: SyntaxWarning: invalid escape sequence '\c'
  sqllite_db_file = "C:\\RanjithC\\AIProjects\\PromptEngg\\huggingface\\data\\simpli_learn\\custsupport\customer_support.db"


True

Building a Feedback Handler

In [7]:
# Create SQLLite DB for Operating on Tickets

def create_sqllite_db() -> sqlite3.Connection:
    
    try:
        sql_conn = sqlite3.connect(sqllite_db_file, check_same_thread=False)    # Tell SQLLite to run on multiple threads.
        
        cursor = sql_conn.cursor()
        cursor.execute(
            """
                CREATE TABLE IF NOT EXISTS support_tickets (
                user_id TEXT, ticket_no INTEGER, status TEXT
                )
            """
        )
        sql_conn.commit()
        cursor.close()

        return(sql_conn)
    
    except Exception as exp:
        print("Creation of SQL DB Failed - {exp}", flush=True)
        return(None)


In [8]:
# Positive & Negative Feedback Handler Services.

# Positive Feedback Handler 
def positive_feedback_handler(user_name: str) -> str:

    response = f"Thank you for your kind words, {user_name}! We're delighted to assist you"
    return(response)

# Negative Feedback Handler
def negative_feedback_handler(user_name: str, sql_conn: sqlite3.Connection) -> str:

    cursor = None

    try:
        ticket_no = random.randint(100000, 999999)

        cursor = sql_conn.cursor()
        cursor.execute("INSERT INTO support_tickets (user_id, ticket_no, status) VALUES (?, ?, ?)",
                    (user_name, ticket_no, "open")
                    )
        sql_conn.commit()

        response = f"We apologize for the inconvenience. A new ticket #{ticket_no} has been generated, and our team will follow up shortly."
        return(response)
    
    except Exception as exp:
        print(f"Error Creating Negative Feedback - {exp}", flush=True)
        return(None)
    finally:
        if (cursor is not None):
            cursor.close()


# List User Tickets
@tool(response_format="content_and_artifact", return_direct=True)
def list_user_tickets(config: RunnableConfig) -> str:
    """ This tool is used to Get List Tickets for a User. 

    Args: 
        config: RunnableConfig Object passed by LLM to the Tool.

    Return: List of Tickets for a User.
    """

    cursor = None
    
    print("Inside List User Tickets", flush=True)

    try:
        user_name = config["configurable"]["user_name"]
        sql_conn = config["configurable"]["sql_conn"]

        cursor = sql_conn.cursor()
        cursor.execute("SELECT ticket_no FROM support_tickets WHERE user_id = ?", 
                       (user_name,)
                      )

        record = cursor.fetchall()
        if (record == None):
            response = f"No Records Found for User: {user_name}."
        else:
            record_list = [row[0] for row in record]
            response = f"Number of Tickets for User: {user_name} - {record_list}"

        return("", {"tool_resp": response})

    except Exception as exp:
        print(f"Error listing User Tickets - {exp}", flush=True)
        return(None)
    finally:
        if (cursor is not None):
            cursor.close()


# Lookup Ticket Status
@tool(response_format="content_and_artifact", return_direct=True)
def lookup_ticket_status(taskinfo: dict, config: RunnableConfig) -> tuple[str,dict]:
    """ This tool is used to Get Status of User's Ticket. 

    Args: 
        taskinfo: Dictionary Object in the below Example format.
            Example - {"ticket": 650932}
        config: RunnableConfig Object passed by LLM to the Tool.

    Return: List of Tickets for a User.
    """

    cursor = None

    print("Inside Ticket Status Lookup", flush=True)

    try:
        user_name = config["configurable"]["user_name"]
        sql_conn = config["configurable"]["sql_conn"]

        ticket_no = taskinfo["ticket"]

        cursor = sql_conn.cursor()
        cursor.execute("SELECT status FROM support_tickets WHERE user_id = ? AND ticket_no = ?", 
                       (user_name, ticket_no)
                      )
    
        record = cursor.fetchone()
        if (record == None):
            response = f"No Records Found for User: {user_name} and TicketNo: {ticket_no}"
        else:
            response = f"Status of your ticket #{ticket_no} is currently marked as: {record[0].upper()}."

        return("", {"tool_resp": response})
    
    except Exception as exp:
        print(f"Error Looking-up Ticket Status - {exp}", flush=True)
        return(None)
    finally:
        if (cursor is not None):
            cursor.close()
    

Build Customer Support Agent

In [9]:
# Create Ollama Agent Chatbot

# Instantiate Customer Support Agent
def create_custsupport_llm(model_name: str) -> ChatOllama:

    ol_custsupport_llm = ChatOllama(
        model = model_name,
        temperature = 0.0,
        num_predict = 512,
        num_ctx = 2048,
        model_kwargs = {
            "repeat_penalty": 1.1,
            "options": {
                "use_cache": False,
                "use_mmap": False
            }
        }   
    )

    return(ol_custsupport_llm)

# Instantiate User Query Agent
def create_userquery_llm(model_name: str) -> ChatOllama:

    ol_userquery_llm = ChatOllama(
        model = model_name,
        temperature = 0.0,
        num_predict = 512,
        num_ctx = 2048,
        model_kwargs = {
            "repeat_penalty": 1.1,
            "options": {
                "use_cache": False,
                "use_mmap": False
            }
        }   
    )

    ol_userquery_llm_with_tools = ol_userquery_llm.bind_tools(
        [
            list_user_tickets,
            lookup_ticket_status
        ]
    )

    return(ol_userquery_llm_with_tools)


In [10]:
# Creating User Query Answering Agent.
# Agent decide which tool to call - User Ticket Status or User Ticket List

def initialise_userquery_agent(userquery_llm: ChatOllama) -> object:

    agent_tools = [list_user_tickets, lookup_ticket_status]

    agent_instruction = """
    Role: You are a User Query Answering Assistant.

    Goals:
    1. Analyze the User Query and decide the right Tool to Call.
   
    Available Tools:
    1. list_user_tickets: Tool used to get/retrieve List of Tickets for a User.
    2. lookup_ticket_status: Tool used to get/lookup Status of a User's Ticket.

    Examples:
        User Input: Could you check the status of ticket 650932 ?
        Assistant: Make ToolCall to 'lookup_ticket_status' Tool.

        User Input: Get the list of my tickets ?
        Assistant: Make ToolCall to 'list_user_tickets' Tool.
    """

    agent = create_agent(
        model = userquery_llm,
        tools = agent_tools,
        system_prompt=SystemMessage(content=agent_instruction)
        )
    
    return(agent)


In [11]:
# Creating an Customer Support Inferencing Agent
# Agent understand the Sentiment of User Input. Positive, Negative, Query

def initialize_custsupport_agent(custsupport_llm: ChatOllama) -> object:

    agent_instruction = """
    Role: You are a Banking Customer Support Assistant.

    Goals:
    1. Analyze the Semantic Meaning of the User Request and CLASSIFY it into EXACTLY ONE of the Allowed TASK.
    2. If the User's intent Matches with Any One Task from the Task List, return that Value.
    3. If the User's intent Does Not Match with value in Task List, respond with UNKNOWN_TASK.
   
    Allowed Tasks:
    1. QueryTask: ONLY IF User Request is to Get Ticket Status or Get List of Tickets.
    2. PositiveFeedback: ONLY IF User is expressing appreciation, satisfaction, or positive sentiment.
    3. NegativeFeedback: ONLY IF User is expressing dissatisfaction, frustration, or a complaint.

    Examples:
        User Input: Thanks for sorting out my net banking login issue.
        Assistant: PositiveFeedback

        User Input: My debit card replacement still hasn't arrived.
        Assistant: NegativeFeedback

        User Input: Could you check the status of ticket 650932 ?
        Assistant: QueryTask

        User Input: What is the Weather today ?
        Assistant: UNKNOWN_TASK
    """

    agent = create_agent(
        model = custsupport_llm,
        system_prompt=SystemMessage(content=agent_instruction)
        )
    
    return(agent)


Initialise & Chat with Agent.

In [12]:
# Format Agent Response

def format_agent_response(ai_msg):
    ai_msg = (ai_msg['messages'][1]).content.split("<|im_start|>assistant")[-1]
    return(ai_msg.strip())

In [ ]:
# Initiate Chat with Agent

exit_flag = False

# Press 'Escape' Key to End the While Loop below
def on_key_press(event):
    if event.name == 'esc':
        print("Escape key pressed! Exiting input.")
        global exit_flag
        exit_flag = True
        return True  # Stop the keyboard listener

# Keyboard Listener
keyboard.on_press(on_key_press)

# Create Customer_Support SQLLite DB
sql_conn = create_sqllite_db()

# Inirialise LLM
llama_model = "llama3.1:8b-instruct-q4_K_M"
custsupport_llm = create_custsupport_llm(llama_model)
userquery_llm = create_userquery_llm(llama_model)

# Initialise Agent
custsupport_agent = initialize_custsupport_agent(custsupport_llm)
userquery_agent = initialise_userquery_agent(userquery_llm)

# Initiate User Conversation & Greetings
user_input = input("Hi, I am a Bank Customer Feedback Agent, Please Enter your First Name: ")
user_name = user_input

# Add User Name to Query Agent Config 
query_agent_config = RunnableConfig(
    configurable = {
        "user_name": user_name,
        "sql_conn": sql_conn
    }
)

# User Interaction Loop.

greeting_flag = True
while(not exit_flag):
    if (greeting_flag == True):
        print(f"Hello {user_name}")
        user_input = input(f"Hello {user_name}, Can I help with Question related to Tickets or Feedback if any: ")
        greeting_flag = False
    else:
        user_input = input("Can an i help with anymore Question realted to Tickets or Feedback: ")

    print("User Input: ", user_input, flush=True)
    if (not exit_flag):
        user_prompt = {"messages": [HumanMessage(content=user_input)]}
        agent_response = custsupport_agent.invoke(user_prompt)
        agent_task = format_agent_response(agent_response)
        if ('PositiveFeedback' in agent_task):                  # PositiveFeedback Processing
            print("Assistant: ", "Thank you for the PositiveFeedback", flush=True)
        elif ('NegativeFeedback' in agent_task):                # NegativeFeedback Processing
            print("Assistant: ", "Sorry to hear the Feedback, Should i create a Support Ticket to resolve your concern ? (Y/N)", flush=True)
            user_confirm = input(f"{user_name} - Create a Support Ticket ? (Y/N)")
            if ((user_confirm.upper() == "Y") or (user_confirm.upper() == "YES")):
                print("Assistant: ", negative_feedback_handler(user_name, sql_conn), flush=True)
        elif ('QueryTask' in agent_task):                       # User Query Processing
            userquery_agent_resp = userquery_agent.invoke(user_prompt, config=query_agent_config)
            userquery_agent_tool_resp = userquery_agent_resp['messages'][-1]
            if isinstance(userquery_agent_tool_resp, ToolMessage):
                print("Assistant: ", userquery_agent_tool_resp.artifact.get("tool_resp"), flush=True)
        else:
            print("Assistant: ", "Not able to understand your Request. I can hlp you with Support Ticket Information or Feedbacks.", flush=True)                 # Unknown Task Processing


User Input:  


In [ ]:
# MANNUAL UPDATE CODE FOR TICKET STATUS

cursor = sql_conn.cursor()
cursor.execute("UPDATE support_tickets SET status = 'CLOSED' WHERE ticket_no = 299247 and user_id = 'Menon'"
            )

sql_conn.commit()
cursor.close()